01 - Build the mosaic corpus
#
# Converts the mounted DICOM series into one mosaic image per selected series:
# 16 slices tiled 4x4, JPEG q95. At 384 that is ~8 GB for the full training
# set, inside the 20 GB notebook-output cap.
#
# JPEG rather than PNG is what pays for the resolution: on already-8-bit
# windowed MRI it is roughly half the size at 0.18% mean error, which is
# nothing next to the 75% of pixels a 256 corpus throws away.
#
# This kernel's output mounts straight into the training notebook as a kernel
# source, so there is no dataset to build by hand. It is resumable: if the
# 12 h session limit hits, re-run and it skips what is already written.

In [ ]:
import sys, subprocess
from pathlib import Path

CODE = next(
    (p for pat in ("rsna-knee-code", "*/rsna-knee-code", "*/*/rsna-knee-code")
     for p in sorted(Path("/kaggle/input").glob(pat)) if (p / "src").is_dir()),
    None,
)
assert CODE is not None, "rsna-knee-code dataset is not attached to this notebook"
sys.path.insert(0, str(CODE / "src"))
sys.path.insert(0, str(CODE))
print("CODE", CODE)

# timm ships in the Kaggle image; the install is a fallback for images that lack
# it, and cannot run in the offline submission notebook.
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=False)
    import timm

import rsna_knee.config as cfg
print("RAW ", cfg.RAW, cfg.RAW.exists())
print("PREP", cfg.PREP, cfg.PREP.exists())
print("SIZE", cfg.SIZE, cfg.MOSAIC_EXT)

In [ ]:
!cd {CODE} && python scripts/build_weak_labels.py

In [ ]:
# --max-minutes stops under Kaggle's session limit so the finished mosaics are
# saved; a killed session can save nothing at all.
!cd {CODE} && python scripts/preprocess_series.py --split train --workers 4 \
    --max-minutes 660

In [ ]:
!cd {CODE} && python scripts/preprocess_series.py --split test --workers 4

In [ ]:
from pathlib import Path
prep = Path("/kaggle/working/prep")
files = [p for p in prep.rglob("*") if p.suffix.lower() in (".jpg", ".png")]
size = sum(p.stat().st_size for p in files)
print(f"{len(files)} mosaics, {size/1e9:.2f} GB")